In [1]:
import pandas as pd

# Read the CSV file into a pandas DataFrame
#df = pd.read_csv('/Users/visjostr/school/parking-prediction/parking-data/Turkuparkkihub_parkings_anon.csv', delimiter=';')
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime objects with specified format
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M')

# Define the time for which you want to check the entries
target_time = pd.to_datetime('03/12/2021 12.00', format='%d/%m/%Y %H.%M')  # Example time, adjust as needed

# Define the parking space ID you want to analyze
parking_space_id = '4b727054-c4dc-492e-8303-0fca60acf0ea'  # Example parking space ID, replace with the actual ID you want to analyze

# Filter the DataFrame for the specific parking space ID
parking_space_df = df[df['region_id'] == parking_space_id]

print(parking_space_df[(parking_space_df['time_start'] <= target_time) & (target_time <= parking_space_df['time_end'])])

# Check the entries at the specified time for the specific parking space
entries_at_time = ((parking_space_df['time_start'] <= target_time) & (target_time <= parking_space_df['time_end'])).sum()

print("Number of entries at", target_time, "for parking space", parking_space_id, ":", entries_at_time)

              created_at       modified_at  \
327018  01/12/2021 16.11  03/12/2021 12.49   
331236  02/12/2021 12.42  03/12/2021 13.50   
332964  02/12/2021 15.16  03/12/2021 12.05   
333005  02/12/2021 15.22  03/12/2021 12.11   
333046  02/12/2021 15.23  03/12/2021 12.44   
...                  ...               ...   
337701  03/12/2021 11.54  03/12/2021 12.30   
337784   03/12/2021 9.36  03/12/2021 12.37   
337786  03/12/2021 11.33  03/12/2021 12.37   
337867  03/12/2021 11.46  03/12/2021 12.45   
338565  03/12/2021 11.58  03/12/2021 13.41   

                                          id  \
327018  7aea6c90-ca7e-422f-b5c8-410b7408ff64   
331236  e2774200-25c3-4f5d-904e-767e0f410beb   
332964  f2348360-8aa8-483f-b36a-5a70801c7208   
333005  c43497e3-72f2-445f-8908-ea85cbfdd605   
333046  ffcd9ec2-9100-4712-b138-1f312b9a24be   
...                                      ...   
337701  cd10df24-4355-4fad-9a71-4ed969214cfb   
337784  80b20710-1b91-453e-bcbf-276f22ebfb35   
337786  2649e6d

In [1]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period


# Function to round datetime to nearest 15 minutes
def round_to_15_minutes(dt):
    reference_date = datetime(2000, 1, 1)  # Choose a reference date for rounding
    rounded = dt - reference_date
    rounded = rounded + timedelta(minutes=7.5)  # Add half interval to round correctly
    rounded -= rounded % timedelta(minutes=15)  # Round to nearest 15 minutes
    return rounded + reference_date

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_15_minutes(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=15)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_unfiltered_15_min.csv', index=False)

# Print time series DataFrame
print(time_series_df)


                     Time  Parkings
324   2021-08-10 10:30:00         1
325   2021-08-10 10:45:00         4
433   2021-08-11 10:30:00         1
434   2021-08-11 10:45:00         1
435   2021-08-11 11:00:00         1
...                   ...       ...
14633 2022-01-21 14:30:00         1
14634 2022-01-21 14:45:00         1
14635 2022-01-21 15:00:00         1
14636 2022-01-21 15:15:00         1
14637 2022-01-21 15:30:00         1

[14638 rows x 2 columns]


In [2]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest minute
def round_to_minute(dt):
    return dt - timedelta(seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_minute(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_minutes_three.csv', index=False)

# Print time series DataFrame
print(time_series_df)


                      Time  Parkings
26724  2021-10-11 00:10:00         1
26725  2021-10-11 00:11:00         1
26726  2021-10-11 00:12:00         1
26727  2021-10-11 00:13:00         1
26728  2021-10-11 00:14:00         1
...                    ...       ...
138226 2022-01-14 23:56:00         1
138227 2022-01-14 23:57:00         1
138228 2022-01-14 23:58:00         1
138229 2022-01-14 23:59:00         1
138230 2022-01-15 00:00:00         1

[138231 rows x 2 columns]


In [4]:
import pandas as pd
from datetime import datetime, timedelta

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest minute
def round_to_minute(dt):
    return dt - timedelta(seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']
    zone_id = row['zone_id']  # Extracting zone_id for each row

    current_time = round_to_minute(time_start)
    while current_time <= time_end:
        key = (current_time, zone_id)  # Using a tuple of (time, zone_id) as key
        if key not in time_series:
            time_series[key] = 0
        time_series[key] += 1
        current_time += timedelta(minutes=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time_Zone', 'Parkings'])

# Splitting 'Time_Zone' column into 'Time' and 'Zone_id'
time_series_df[['Time', 'Zone_id']] = pd.DataFrame(time_series_df['Time_Zone'].tolist(), index=time_series_df.index)
time_series_df.drop(columns=['Time_Zone'], inplace=True)

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Export DataFrame to a CSV file
#time_series_df.to_csv('parking_time_series_minutes_three.csv', index=False)

# Print time series DataFrame
print(time_series_df)


        Parkings                Time  Zone_id
44878          1 2021-10-11 00:10:00        2
44879          1 2021-10-11 00:11:00        2
44880          1 2021-10-11 00:12:00        2
44881          1 2021-10-11 00:13:00        2
44882          1 2021-10-11 00:14:00        2
...          ...                 ...      ...
551586         1 2022-01-14 23:56:00        2
551587         1 2022-01-14 23:57:00        2
551588         1 2022-01-14 23:58:00        2
551589         1 2022-01-14 23:59:00        2
551590         1 2022-01-15 00:00:00        2

[551643 rows x 3 columns]


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import holidays

# Read the CSV file into a DataFrame
df = pd.read_csv('../parking-data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest hour
def round_to_nearest_hour(dt):
    if pd.isnull(dt):
        return dt
    reference_date = datetime(2000, 1, 1)  # Choose a reference date for rounding
    rounded = dt - reference_date
    rounded = rounded + timedelta(minutes=30)  # Add half interval to round correctly
    rounded -= rounded % timedelta(hours=1)  # Round to nearest hour
    return rounded + reference_date

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_nearest_hour(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(hours=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)


# Extract additional features
time_series_df['day_of_year'] = time_series_df['Time'].dt.dayofyear
time_series_df['weekday'] = time_series_df['Time'].dt.weekday + 1  # Monday=1, Sunday=7
time_series_df['hour'] = time_series_df['Time'].dt.hour

# Generate holiday dates for the relevant years (assuming Finland's holidays)
finland_holidays = holidays.Finland(years=time_series_df['Time'].dt.year.unique())

# Define a function to check if a date is a holiday or Sunday
def is_holiday_or_sunday(date):
    return date in finland_holidays or date.weekday() == 6  # Sunday is 6 in datetime

# Apply the function to create the is_holiday column
time_series_df['is_holiday'] = time_series_df['Time'].apply(lambda x: 1 if is_holiday_or_sunday(x) else 0)


# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_with_features_1_hour.csv', index=False)


# Print time series DataFrame
print(time_series_df)


                    Time  Parkings  day_of_year  weekday  hour  is_holiday
463  2021-10-11 00:00:00         1          284        1     0           0
464  2021-10-11 01:00:00         1          284        1     1           0
465  2021-10-11 02:00:00         2          284        1     2           0
466  2021-10-11 03:00:00         2          284        1     3           0
467  2021-10-11 04:00:00         2          284        1     4           0
...                  ...       ...          ...      ...   ...         ...
2300 2022-01-14 20:00:00        38           14        5    20           0
2301 2022-01-14 21:00:00        14           14        5    21           0
2302 2022-01-14 22:00:00         8           14        5    22           0
2303 2022-01-14 23:00:00         4           14        5    23           0
2304 2022-01-15 00:00:00         1           15        6     0           0

[2305 rows x 6 columns]


In [1]:
# With features 1 minute interval

import pandas as pd
from datetime import datetime, timedelta
import holidays

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest minute
def round_to_minute(dt):
    return dt - timedelta(seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_minute(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=1)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Extract additional features
time_series_df['day_of_year'] = time_series_df['Time'].dt.dayofyear
time_series_df['weekday'] = time_series_df['Time'].dt.weekday + 1  # Monday=1, Sunday=7
time_series_df['hour'] = time_series_df['Time'].dt.hour

# Generate holiday dates for the relevant years (assuming Finland's holidays)
finland_holidays = holidays.Finland(years=time_series_df['Time'].dt.year.unique())

# Define a function to check if a date is a holiday or Sunday
def is_holiday_or_sunday(date):
    return date in finland_holidays or date.weekday() == 6  # Sunday is 6 in datetime

# Apply the function to create the is_holiday column
time_series_df['is_holiday'] = time_series_df['Time'].apply(lambda x: 1 if is_holiday_or_sunday(x) else 0)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_with_features.csv', index=False)

# Print the final DataFrame
print(time_series_df.head())


                     Time  Parkings  day_of_year  weekday  hour  is_holiday
26724 2021-10-11 00:10:00         1          284        1     0           0
26725 2021-10-11 00:11:00         1          284        1     0           0
26726 2021-10-11 00:12:00         1          284        1     0           0
26727 2021-10-11 00:13:00         1          284        1     0           0
26728 2021-10-11 00:14:00         1          284        1     0           0


In [ ]:
# With features 15 minute interval

import pandas as pd
from datetime import datetime, timedelta
import holidays

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest 15 minutes
def round_to_nearest_15min(dt):
    return dt - timedelta(minutes=dt.minute % 15, seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_nearest_15min(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=15)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Extract additional features
time_series_df['day_of_year'] = time_series_df['Time'].dt.dayofyear
time_series_df['weekday'] = time_series_df['Time'].dt.weekday + 1  # Monday=1, Sunday=7
time_series_df['hour'] = time_series_df['Time'].dt.hour

# Generate holiday dates for the relevant years (assuming Finland's holidays)
finland_holidays = holidays.Finland(years=time_series_df['Time'].dt.year.unique())

# Define a function to check if a date is a holiday or Sunday
def is_holiday_or_sunday(date):
    return date in finland_holidays or date.weekday() == 6  # Sunday is 6 in datetime

# Apply the function to create the is_holiday column
time_series_df['is_holiday'] = time_series_df['Time'].apply(lambda x: 1 if is_holiday_or_sunday(x) else 0)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_with_features_15min.csv', index=False)

# Print the final DataFrame
print(time_series_df.head())


                    Time  Parkings  day_of_year  weekday  hour  is_holiday
1818 2021-10-11 00:00:00         1          284        1     0           0
1819 2021-10-11 00:15:00         1          284        1     0           0
1820 2021-10-11 00:30:00         1          284        1     0           0
1821 2021-10-11 00:45:00         1          284        1     0           0
1822 2021-10-11 01:00:00         1          284        1     1           0


In [1]:


#print(df.terminal_number.mode())
#print(df.zone_id.nunique())
#print(df[df['zone_id'] == 1].region_id.unique())
print(df[df['is_disc_parking'] == '1'].is_disc_parking)
print(df.is_disc_parking.nunique())

print(df[df['parking_area_id'] == '52c37d3d-2a28-4bde-8996-fdff25e4157d'])

NameError: name 'df' is not defined

In [ ]:
# With features 15 minute interval

import pandas as pd
from datetime import datetime, timedelta
import holidays

# Read the CSV file into a DataFrame
df = pd.read_csv('../data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest 15 minutes
def round_to_nearest_15min(dt):
    return dt - timedelta(minutes=dt.minute % 15, seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_nearest_15min(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=15)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Extract additional features
time_series_df['day_of_year'] = time_series_df['Time'].dt.dayofyear
time_series_df['weekday'] = time_series_df['Time'].dt.weekday + 1  # Monday=1, Sunday=7
time_series_df['hour'] = time_series_df['Time'].dt.hour

# Generate holiday dates for the relevant years (assuming Finland's holidays)
finland_holidays = holidays.Finland(years=time_series_df['Time'].dt.year.unique())

# Define a function to check if a date is a holiday or Sunday
def is_holiday_or_sunday(date):
    return date in finland_holidays or date.weekday() == 6  # Sunday is 6 in datetime

# Apply the function to create the is_holiday column
time_series_df['is_holiday'] = time_series_df['Time'].apply(lambda x: 1 if is_holiday_or_sunday(x) else 0)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_with_features_15min.csv', index=False)

# Print the final DataFrame
print(time_series_df.head())


In [3]:
import pandas as pd

# Load the dataset
df = pd.read_csv('../parking-data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Get the top 15 most entries of 'parking_area_id'
top_15_parking_areas = df['parking_area_id'].value_counts().head(15)

print(top_15_parking_areas)


parking_area_id
52c37d3d-2a28-4bde-8996-fdff25e4157d    27527
6a9b8e6d-5f15-48ee-9ef9-51cd4d902b68    25703
72ff5003-9b0f-4a82-a1e7-8f669aaf21c7    17049
72a9a0ee-ee6b-4e4e-8c94-12fb30f8e903    14475
0e7b37f9-5b0d-426e-a96a-4c9c7332dafd    13437
f9edb154-57ec-444b-92b3-6331a8141048    12600
3283246c-a1f0-4a44-b06f-501c69906269    12185
15a4549f-6bbb-490f-a57c-927da72b99a6    12121
2e2109a9-608b-45b9-9513-2b3406c4925d    11635
5c90c71c-6ac4-4e45-ae19-6c4dfd759675    10611
e1556705-1210-42bf-a7e2-c073865a332a    10336
eee26c46-8870-474c-936d-d77b5e8a8c5c     9832
1683e849-6427-4580-b7b7-4126c727a086     9108
eee2cac7-c159-4d7b-abcf-2fcb083770d4     8906
20032cb9-7017-452a-8a8b-7d6a8bb732dd     8710
Name: count, dtype: int64


In [7]:
import pandas as pd
from datetime import datetime, timedelta
import holidays

# Read the CSV file into a DataFrame
df = pd.read_csv('../parking-data/Turkuparkkihub_parkings_anon.csv', delimiter=';')

# Filter the DataFrame to only include the specified parking area
df = df[df['parking_area_id'] == '52c37d3d-2a28-4bde-8996-fdff25e4157d']

# Convert 'time_start' and 'time_end' columns to datetime type
df['time_start'] = pd.to_datetime(df['time_start'], format='%d/%m/%Y %H.%M', errors='coerce')
df['time_end'] = pd.to_datetime(df['time_end'], format='%d/%m/%Y %H.%M', errors='coerce')

# Filter the DataFrame to cover the specified time period
start_date = datetime(2021, 10, 11)
end_date = datetime(2022, 1, 15)
df_filtered = df[(df['time_start'] >= start_date) & (df['time_end'] <= end_date)]

# Function to round datetime to nearest 15 minutes
def round_to_nearest_15min(dt):
    return dt - timedelta(minutes=dt.minute % 15, seconds=dt.second, microseconds=dt.microsecond)

# Initialize time series data
time_series = {}

# Iterate through each row in the filtered DataFrame
for index, row in df_filtered.iterrows():
    time_start = row['time_start']
    time_end = row['time_end']

    current_time = round_to_nearest_15min(time_start)
    while current_time <= time_end:
        if current_time not in time_series:
            time_series[current_time] = 0
        time_series[current_time] += 1
        current_time += timedelta(minutes=15)

# Convert time series data to DataFrame
time_series_df = pd.DataFrame(list(time_series.items()), columns=['Time', 'Parkings'])

# Sort DataFrame in ascending order based on the datetime column
time_series_df = time_series_df.sort_values(by='Time', ascending=True)

# Extract additional features
time_series_df['day_of_year'] = time_series_df['Time'].dt.dayofyear
time_series_df['weekday'] = time_series_df['Time'].dt.weekday + 1  # Monday=1, Sunday=7
time_series_df['hour'] = time_series_df['Time'].dt.hour

# Generate holiday dates for the relevant years (assuming Finland's holidays)
finland_holidays = holidays.Finland(years=time_series_df['Time'].dt.year.unique())

# Define a function to check if a date is a holiday or Sunday
def is_holiday_or_sunday(date):
    return date in finland_holidays or date.weekday() == 6  # Sunday is 6 in datetime

# Apply the function to create the is_holiday column
time_series_df['is_holiday'] = time_series_df['Time'].apply(lambda x: 1 if is_holiday_or_sunday(x) else 0)

# Export DataFrame to a CSV file
time_series_df.to_csv('parking_time_series_with_features_15min_one_spot.csv', index=False)

# Print the final DataFrame
print(time_series_df.head())


                  Time  Parkings  day_of_year  weekday  hour  is_holiday
26 2021-10-11 08:00:00         1          284        1     8           0
27 2021-10-11 08:15:00         2          284        1     8           0
28 2021-10-11 08:30:00         2          284        1     8           0
29 2021-10-11 08:45:00         1          284        1     8           0
30 2021-10-11 09:00:00         2          284        1     9           0
